# 4. RL Training

Train the RL agent using Maskable PPO.

## 4.0 Install Dependencies

In [1]:
# Install required packages (safe to re-run; --quiet suppresses noise)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
    'numpy',
    'pandas',
    'joblib',
    'gymnasium',
    'stable-baselines3',
    'sb3-contrib',
    'tqdm',
    'rich',
    'sentence-transformers',
])
# Install CPU-only torch (use the CUDA index URL if you have a GPU):
# GPU:  --index-url https://download.pytorch.org/whl/cu121
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
    'torch', '--index-url', 'https://download.pytorch.org/whl/cpu',
])


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


0

## 4.1 Setup

In [2]:
# ── Google Colab / Google Drive setup ────────────────────────────────────────
# When running on Colab, mount your Drive and set DRIVE_REPO_PATH to the folder
# where you cloned / uploaded this repo (e.g. '/content/drive/MyDrive/bwa').
# Leave DRIVE_REPO_PATH as None when running locally — it will be ignored.
import os, sys
from pathlib import Path

DRIVE_REPO_PATH = '/content/drive/MyDrive/bwa'   # ← change to your Drive path when on Colab
                         #   e.g. '/content/drive/MyDrive/bwa'

ON_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if ON_COLAB and DRIVE_REPO_PATH is not None:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    REPO_ROOT = Path(DRIVE_REPO_PATH)
    print(f'Colab + Drive: repo root → {REPO_ROOT}')
else:
    # Local: walk up from cwd until we find src/data_ingestion.py
    REPO_ROOT = None
    for _p in [Path.cwd(), *Path.cwd().parents]:
        if (_p / 'src' / 'data_ingestion.py').exists():
            REPO_ROOT = _p
            break
    if REPO_ROOT is None:
        raise RuntimeError('Could not locate repo root. Set DRIVE_REPO_PATH or run from inside the repo.')
    print(f'Local: repo root → {REPO_ROOT}')

_src = str(REPO_ROOT / 'src')
if _src not in sys.path:
    sys.path.insert(0, _src)

DATASET   = 'BPIC2012'
OUTPUT_DIR = REPO_ROOT / 'output' / DATASET
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'OUTPUT_DIR → {OUTPUT_DIR}')

Local: repo root → /mnt/hdd/Code/Git/bwa
OUTPUT_DIR → /mnt/hdd/Code/Git/bwa/output/BPIC2012


## 4.1 Load Twin & Embeddings

In [3]:
import joblib
twin = joblib.load(OUTPUT_DIR / f'digital_twin_{DATASET}_train.pkl')
embedder = joblib.load(OUTPUT_DIR / f'activity_embeddings.model')
print(f"Loaded twin: {len(twin.activities)} activities")

Loaded twin: 24 activities


## 4.2 Setup Env

In [4]:
import gymnasium as gym
from rl_env import ProcessEnv
from kpi_actions import MANAGEMENT_ACTIONS, N_MANAGEMENT_ACTIONS, RULE_DESCRIPTIONS
import json

# Load pre-computed terminal classification from notebook 02
_tc_path = OUTPUT_DIR / 'terminal_classification.json'
if _tc_path.exists():
    with open(_tc_path) as _f:
        _tc = json.load(_f)
    bad_terminals = set(_tc['bad_terminals'])
    print(f'Loaded terminal classification: bad={sorted(bad_terminals)}')
else:
    bad_terminals = None  # ProcessEnv will run classify_bad_terminals() as fallback
    print('terminal_classification.json not found — run notebook 02 first')

env = ProcessEnv(twin=twin, embed_model=embedder, kpi_baselines=twin.kpi_baselines,
                 bad_terminals=bad_terminals)

# Action space is now MultiDiscrete([max_successors, N_MANAGEMENT_ACTIONS])
print(f'Action space: {env.action_space}')
print(f'  Routing actions  (dim 0): {env._max_succ}  — one slot per possible next activity')
print(f'  Management actions (dim 1): {N_MANAGEMENT_ACTIONS}  — KPI-based interventions')
print(f'Observation keys: {list(env.observation_space.spaces.keys())}')
print()
print('Management actions available:')
for a in MANAGEMENT_ACTIONS:
    print(f'  [{a.index:2d}] {a.name:<40s}  {a.description}')


Loaded terminal classification: bad=['A_CANCELLED', 'A_DECLINED', 'O_CANCELLED', 'O_DECLINED']
Action space: MultiDiscrete([24 15])
  Routing actions  (dim 0): 24  — one slot per possible next activity
  Management actions (dim 1): 15  — KPI-based interventions
Observation keys: ['case_embedding', 'kpi_signals', 'mgmt_action_mask', 'resource_state']

Management actions available:
  [ 0] assign_to_primary_team                    Default manager action for active non-terminal cases.
  [ 1] outsource_to_volunteer_pool               Valid when workload/volume pressure and delay are both high.
  [ 2] rebalance_overloaded_queue                Valid when rework or queue-delay suggests local overload.
  [ 3] merge_tasks_under_role                    Valid when volume pressure is high and risk is not high.
  [ 4] prioritize_urgent_case                    Valid when case-age is high or risk branch indicates urgency.
  [ 5] defer_until_objections_resolved           Valid when objection/appeal sig

## 4.3 Reward Weight Tuning

Searches for reward weights that produce KPI outcomes matching the real log.
Runs ~60 short rollouts under a random policy and scores each weight vector
by how closely the achieved completion rate and rework match the real log targets.

C2 env weight mapping:
- `w_completion` → `w_terminal` (bonus for reaching terminal activity)
- `w_rework`     → `w_loop`     (penalty per excess routing loop)
- `w_delay`      → `w_progress` (bonus for moving toward terminal)
- `w_throughput` → `w_step`     (per-step cost to incentivise efficiency)

Takes ~2–3 minutes. Skip and re-run training if you want to use the defaults.

In [5]:
import pandas as pd
import json
from reward_tuning import tune_reward_weights

weights_path = OUTPUT_DIR / 'reward_weights.json'

if weights_path.exists():
    # Reload previously tuned weights — skip re-tuning
    with open(weights_path) as _f:
        best_weights = json.load(_f)
    # Apply to env
    env.w_terminal = best_weights.get('w_completion', env.w_terminal)
    env.w_loop     = best_weights.get('w_rework',     env.w_loop)
    env.w_progress = best_weights.get('w_delay',      env.w_progress)
    env.w_step     = best_weights.get('w_throughput',  env.w_step)
    print(f'Loaded saved weights from {weights_path}')
    print(f'  w_terminal={env.w_terminal}  w_loop={env.w_loop}  '
          f'w_progress={env.w_progress}  w_step={env.w_step}')
else:
    # Load pre-computed KPIs from notebook 02 (avoids re-reading the full event log)
    from feature_engineering import compute_case_kpis
    kpi_path = OUTPUT_DIR / f'kpis_{DATASET}_train.parquet'
    if kpi_path.exists():
        kpi_df = pd.read_parquet(kpi_path)
        print(f'Loaded KPIs from {kpi_path}')
    else:
        # Fallback: recompute from raw events
        print('kpis parquet not found — recomputing from raw events (run notebook 02 first)')
        df_real = pd.read_parquet(OUTPUT_DIR / f'events_{DATASET}_train.parquet')
        kpi_df  = compute_case_kpis(df_real)

    # tune_reward_weights handles MultiDiscrete action space automatically —
    # it detects env.action_space type and samples both routing + management
    # actions from their respective valid masks during rollouts.
    best_weights = tune_reward_weights(
        env=env,
        kpi_df=kpi_df,
        n_episodes_per_trial=40,
        random_search=True,
        n_random_trials=60,
        seed=42,
        verbose=True,
    )

    # Save so next run skips tuning
    with open(weights_path, 'w') as _f:
        json.dump(best_weights, _f, indent=2)
    print(f'Saved tuned weights to {weights_path}')


Loaded saved weights from /mnt/hdd/Code/Git/bwa/output/BPIC2012/reward_weights.json
  w_terminal=15.0  w_loop=0.5  w_progress=1.0  w_step=0.02


## 4.4 Train

In [6]:
# Force unbuffered stdout so log lines appear live in Jupyter
import os, sys
os.environ['PYTHONUNBUFFERED'] = '1'

from sb3_contrib import MaskablePPO
from stable_baselines3.common.callbacks import CallbackList
from training_logger import TrainingLogger
from early_stopping import EarlyStoppingCallback

LOG_INTERVAL = 10_000   # log + early-stopping check every N steps
MAX_STEPS    = 500_000  # hard cap — early stopping will usually kick in sooner

early_stop = EarlyStoppingCallback(
    window=5,          # smooth over last 5 intervals (50k steps)
    min_delta=0.5,     # must improve by at least 0.5 reward to count as progress
    patience=5,        # stop after 5 consecutive non-improving checks
    check_freq=LOG_INTERVAL,
    verbose=1,
)
logger = TrainingLogger(
    log_interval=LOG_INTERVAL,
    log_path=str(OUTPUT_DIR / 'rl_model'),
    early_stopping=early_stop,  # wires the two callbacks together
)

# MaskablePPO with MultiInputPolicy handles MultiDiscrete action spaces natively.
# The combined action mask (routing | management) is passed via action_masks().
model = MaskablePPO('MultiInputPolicy', env, verbose=0)
print(f'Training up to {MAX_STEPS:,} steps (early stopping enabled)...')
print(f'Action space: {env.action_space}  (routing + management)')
model.learn(total_timesteps=MAX_STEPS, callback=CallbackList([logger, early_stop]))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save(OUTPUT_DIR / 'rl_model/best_model.zip')
print('Model saved')


Training up to 500,000 steps (early stopping enabled)...
Action space: MultiDiscrete([24 15])  (routing + management)
──────────────────────────────────────────────────────────────
    step    eps       rew       ±   term      H      ev    fps
──────────────────────────────────────────────────────────────
   10000   4062    +13.72    3.80  100%   2.21   0.018    660
  [EarlyStopping] step=10000  warming up (1/5 intervals)
   20000   7186    +16.15    2.74  100%   2.32   0.072    587
  [EarlyStopping] step=20000  warming up (2/5 intervals)
   30000   8220    +20.02    3.25  100%   2.42   0.416    607
  [EarlyStopping] step=30000  warming up (3/5 intervals)
   40000   8963    +22.26    2.65  100%   2.25   0.634    595
  [EarlyStopping] step=40000  warming up (4/5 intervals)
   50000   9610    +23.43    1.94  100%   2.11   0.791    587
   60000  10226    +24.09    1.67  100%   2.04   0.840    574
   70000  10790    +24.94    1.85  100%   2.02   0.789    600
   80000  11308    +26.22    2.

## 4.4 Training Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
import numpy as np

# TrainingLogger writes to training_metrics.csv (not SB3's progress.csv)
metrics_file = OUTPUT_DIR / 'rl_model' / 'training_metrics.csv'

# ── Style ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'axes.facecolor': '#f8f9fa', 'figure.facecolor': 'white',
    'axes.grid': True, 'grid.color': '#dee2e6', 'grid.linewidth': 0.6,
    'axes.spines.top': False, 'axes.spines.right': False,
})

if not metrics_file.exists():
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.text(0.5, 0.5, 'No training data yet — run the Train cell first.',
            ha='center', va='center', fontsize=13, color='#7f8c8d',
            transform=ax.transAxes)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    df = pd.read_csv(metrics_file)
    steps = df['timestep']

    def smooth(s, w=3):
        # Simple rolling mean for visual clarity
        return s.rolling(w, min_periods=1).mean()

    # Check whether management action columns are present
    mgmt_cols = [c for c in df.columns if c.startswith('mgmt_')]
    has_mgmt = len(mgmt_cols) > 0

    n_rows = 3 if has_mgmt else 2
    fig = plt.figure(figsize=(16, 5 * n_rows))
    gs  = gridspec.GridSpec(n_rows, 3, figure=fig, hspace=0.50, wspace=0.35)

    # ── 1. Episode reward ─────────────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, :])
    ax1.fill_between(steps,
                     df['ep_reward_mean'] - df['ep_reward_std'],
                     df['ep_reward_mean'] + df['ep_reward_std'],
                     alpha=0.15, color='#3498db')
    ax1.plot(steps, df['ep_reward_mean'], color='#3498db', lw=2, label='Mean reward')
    ax1.plot(steps, smooth(df['ep_reward_mean']), color='#2980b9',
             lw=1.5, ls='--', alpha=0.7, label='Smoothed')
    ax1.axhline(0, color='#bdc3c7', lw=0.8, ls=':')
    ax1.set_xlabel('Timesteps', fontsize=10)
    ax1.set_ylabel('Episode Reward', fontsize=10)
    ax1.set_title('Episode Reward (mean ± std)', fontsize=11, fontweight='bold')
    ax1.legend(fontsize=9)

    # ── 2. Terminal rate ──────────────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.plot(steps, df['terminal_rate'] * 100, color='#2ecc71', lw=2)
    ax2.set_xlabel('Timesteps', fontsize=9)
    ax2.set_ylabel('Terminal Rate (%)', fontsize=9)
    ax2.set_title('Episode Completion Rate', fontsize=10, fontweight='bold')
    ax2.set_ylim(0, 105)

    # ── 3. KPI signals + episode length ──────────────────────────────────────
    ax3 = fig.add_subplot(gs[1, 1])
    ax3.plot(steps, df['delay_mean'],  color='#e74c3c', lw=1.5, label='Delay')
    ax3.plot(steps, df['rework_mean'], color='#f39c12', lw=1.5, label='Rework')
    if 'ep_len_mean' in df.columns:
        ax3b = ax3.twinx()
        ax3b.plot(steps, df['ep_len_mean'], color='#3498db', lw=1, ls=':', alpha=0.7, label='Ep length')
        ax3b.set_ylabel('Episode Length', fontsize=9, color='#3498db')
        ax3b.spines['right'].set_visible(True)
        lines1, labels1 = ax3.get_legend_handles_labels()
        lines2, labels2 = ax3b.get_legend_handles_labels()
        ax3.legend(lines1 + lines2, labels1 + labels2, fontsize=8)
    else:
        ax3.legend(fontsize=8)
    ax3.set_xlabel('Timesteps', fontsize=9)
    ax3.set_ylabel('KPI (normalised)', fontsize=9)
    ax3.set_title('KPI Signals + Episode Length', fontsize=10, fontweight='bold')

    # ── 4. Action entropy + value loss ────────────────────────────────────────
    ax4 = fig.add_subplot(gs[1, 2])
    ax4.plot(steps, df['action_entropy'], color='#1abc9c', lw=1.5, label='Action entropy')
    ax4.set_xlabel('Timesteps', fontsize=9)
    ax4.set_ylabel('Entropy', fontsize=9, color='#1abc9c')
    ax4.set_title('Exploration (Entropy)', fontsize=10, fontweight='bold')
    if df['value_loss'].notna().any():
        ax4b = ax4.twinx()
        ax4b.plot(steps, df['value_loss'], color='#e67e22', lw=1, ls='--',
                  alpha=0.7, label='Value loss')
        ax4b.set_ylabel('Value Loss', fontsize=9, color='#e67e22')
        ax4b.spines['right'].set_visible(True)
        lines1, labels1 = ax4.get_legend_handles_labels()
        lines2, labels2 = ax4b.get_legend_handles_labels()
        ax4.legend(lines1 + lines2, labels1 + labels2, fontsize=8)
    else:
        ax4.legend(fontsize=8)

    # ── 5. Management action usage (if logged) ────────────────────────────────
    if has_mgmt:
        from kpi_actions import MANAGEMENT_ACTIONS
        ax5 = fig.add_subplot(gs[2, :])
        cmap = plt.get_cmap('tab20')
        for i, col in enumerate(mgmt_cols):
            action_name = col.replace('mgmt_', '').replace('_rate', '')
            ax5.plot(steps, df[col] * 100, lw=1.5,
                     color=cmap(i / max(len(mgmt_cols), 1)),
                     label=action_name, alpha=0.85)
        ax5.set_xlabel('Timesteps', fontsize=9)
        ax5.set_ylabel('Usage Rate (%)', fontsize=9)
        ax5.set_title('Management Action Usage Rate over Training', fontsize=10, fontweight='bold')
        ax5.legend(fontsize=7, ncol=3, loc='upper right')

    fig.suptitle(f'Training Progress — {DATASET}  ({int(steps.max()):,} steps)',
                 fontsize=13, fontweight='bold', y=1.01)

    plt.savefig(OUTPUT_DIR / 'training_progress.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved to {OUTPUT_DIR / "training_progress.png"}')
